# Task 2A: Classification - Mental Health Mood Prediction

**Goal:** Predict next-day mood as a classification problem by discretizing mood into classes.

We use two fundamentally different classifier types:
1. **Random Forest** - a tabular classifier on the feature-engineered dataset
2. **LSTM** - a temporal/sequential classifier that leverages time-ordered data

Both are evaluated with a time-based train/test split to respect the temporal nature of the data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler, LabelEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

print("All imports loaded successfully.")

## 1. Load Data and Define Mood Classes

In [ ]:
# Load the feature-engineered dataset from Task 1C
df = pd.read_csv('../data/dataset_features.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
df.head()

In [ ]:
# --- Identify key columns ---
# TODO: Adjust these column names to match your actual dataset_features.csv
# Expected columns: a patient/id column, a date/time column, feature columns, and a target mood column.

# Set these to match your dataset:
ID_COL = 'id'          # patient identifier column
DATE_COL = 'date'      # date column (for temporal ordering)
TARGET_COL = 'mood'    # target: next-day average mood (continuous, to be discretized)

# All other columns are features
FEATURE_COLS = [c for c in df.columns if c not in [ID_COL, DATE_COL, TARGET_COL, '']]
print(f"Number of features: {len(FEATURE_COLS)}")
print(f"Target column: {TARGET_COL}")
print(f"\nTarget distribution (continuous):")
print(df[TARGET_COL].describe())

In [ ]:
# --- Discretize mood into classes ---
# We use tercile-based binning (3 classes): Low, Medium, High
# This ensures roughly balanced classes.

mood_values = df[TARGET_COL].dropna()
q33 = mood_values.quantile(0.33)
q66 = mood_values.quantile(0.66)
print(f"Tercile thresholds: Low < {q33:.2f}, Medium [{q33:.2f}, {q66:.2f}), High >= {q66:.2f}")

def discretize_mood(value):
    """Convert continuous mood to categorical class."""
    if pd.isna(value):
        return np.nan
    if value < q33:
        return 'Low'
    elif value < q66:
        return 'Medium'
    else:
        return 'High'

df['mood_class'] = df[TARGET_COL].apply(discretize_mood)

# Show class distribution
print(f"\nMood class distribution:")
print(df['mood_class'].value_counts().sort_index())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET_COL].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(q33, color='red', linestyle='--', label=f'33rd pctl = {q33:.2f}')
axes[0].axvline(q66, color='orange', linestyle='--', label=f'66th pctl = {q66:.2f}')
axes[0].set_xlabel('Mood (continuous)')
axes[0].set_ylabel('Count')
axes[0].set_title('Mood Distribution with Class Boundaries')
axes[0].legend()

df['mood_class'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color=['#e74c3c', '#f39c12', '#27ae60'])
axes[1].set_title('Mood Class Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

## 2. Evaluation Setup

### Why time-based splitting?
Since our data is temporal (daily observations per patient), we cannot use random train/test splits. Random splitting would cause **data leakage**: the model could learn from future observations to predict past ones, which is unrealistic in deployment.

Instead, we use a **chronological split**: the first ~80% of time points for training, the last ~20% for testing. For cross-validation within training, we use `TimeSeriesSplit` which respects temporal ordering.

### Metrics
- **Accuracy** - overall correctness
- **F1 (macro)** - balances precision and recall across all classes equally, important when classes may be slightly imbalanced
- **Confusion matrix** - reveals which mood classes are confused with each other

In [ ]:
# --- Prepare data and time-based split ---

# Drop rows with missing target
df_clean = df.dropna(subset=['mood_class']).copy()

# Ensure date column is datetime and sort by date
df_clean[DATE_COL] = pd.to_datetime(df_clean[DATE_COL])
df_clean = df_clean.sort_values([DATE_COL, ID_COL]).reset_index(drop=True)

# Encode target
le = LabelEncoder()
df_clean['mood_label'] = le.fit_transform(df_clean['mood_class'])
class_names = le.classes_
print(f"Class encoding: {dict(zip(class_names, le.transform(class_names)))}")

# Drop any remaining NaN in features
df_clean = df_clean.dropna(subset=FEATURE_COLS)

# Time-based split: first 80% of dates for train, last 20% for test
unique_dates = df_clean[DATE_COL].sort_values().unique()
split_idx = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_idx]

train_mask = df_clean[DATE_COL] < split_date
test_mask = df_clean[DATE_COL] >= split_date

X_train = df_clean.loc[train_mask, FEATURE_COLS].values
y_train = df_clean.loc[train_mask, 'mood_label'].values
X_test = df_clean.loc[test_mask, FEATURE_COLS].values
y_test = df_clean.loc[test_mask, 'mood_label'].values

print(f"\nSplit date: {split_date}")
print(f"Train set: {X_train.shape[0]} samples ({train_mask.sum() / len(df_clean) * 100:.1f}%)")
print(f"Test set:  {X_test.shape[0]} samples ({test_mask.sum() / len(df_clean) * 100:.1f}%)")
print(f"\nTrain class distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test class distribution:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

In [ ]:
# --- Scale features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled train features shape: {X_train_scaled.shape}")
print(f"Scaled test features shape:  {X_test_scaled.shape}")

## 3. Classifier 1: Random Forest (Tabular Approach)

Random Forest is a strong baseline for tabular data. It handles non-linear relationships, is robust to outliers, and provides feature importance scores. We tune hyperparameters using `GridSearchCV` with `TimeSeriesSplit` as the cross-validation strategy.

In [ ]:
# --- Hyperparameter tuning with TimeSeriesSplit ---
tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', None]
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=tscv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1 (macro): {grid_search.best_score_:.4f}")

In [ ]:
# --- Evaluate Random Forest on test set ---
rf_best = grid_search.best_estimator_
y_pred_rf = rf_best.predict(X_test_scaled)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf, average='macro')

print("=== Random Forest - Test Set Results ===")
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"F1 (macro): {rf_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=class_names))

In [ ]:
# --- Confusion matrix for Random Forest ---
fig, ax = plt.subplots(figsize=(6, 5))
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Random Forest - Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature importance ---
importances = rf_best.feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values(ascending=True)

# Show top 20 features
top_n = min(20, len(feat_imp))
fig, ax = plt.subplots(figsize=(8, max(6, top_n * 0.35)))
feat_imp.tail(top_n).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Random Forest - Top {top_n} Feature Importances')
ax.set_xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

## 4. Classifier 2: LSTM (Temporal Approach)

The LSTM (Long Short-Term Memory) network is inherently temporal - it processes sequences of daily feature vectors per patient. This allows the model to learn temporal patterns and dependencies that a tabular classifier cannot capture.

**Data preparation:** For each patient, we create sliding windows of consecutive days as input sequences, with the mood class of the next day as the target.

In [ ]:
# --- Prepare sequential data for LSTM ---
SEQUENCE_LENGTH = 7  # use past 7 days to predict next-day mood class

def create_sequences(dataframe, feature_cols, label_col, id_col, date_col, seq_len):
    """Create sliding window sequences per patient."""
    sequences = []
    labels = []
    
    for patient_id in dataframe[id_col].unique():
        patient_data = dataframe[dataframe[id_col] == patient_id].sort_values(date_col)
        features = patient_data[feature_cols].values
        targets = patient_data[label_col].values
        
        for i in range(len(features) - seq_len):
            seq = features[i:i + seq_len]
            label = targets[i + seq_len]  # predict mood class at end of window
            sequences.append(seq)
            labels.append(label)
    
    return np.array(sequences), np.array(labels)

# Split into train/test based on dates, then create sequences per patient
df_train_lstm = df_clean[df_clean[DATE_COL] < split_date].copy()
df_test_lstm = df_clean[df_clean[DATE_COL] >= split_date].copy()

# Scale features per the same scaler
df_train_lstm[FEATURE_COLS] = scaler.transform(df_train_lstm[FEATURE_COLS])
df_test_lstm[FEATURE_COLS] = scaler.transform(df_test_lstm[FEATURE_COLS])

X_train_seq, y_train_seq = create_sequences(
    df_train_lstm, FEATURE_COLS, 'mood_label', ID_COL, DATE_COL, SEQUENCE_LENGTH
)

# For test sequences, we need data from both train and test periods
# (the first SEQUENCE_LENGTH days of test need prior context)
df_all_lstm = pd.concat([df_train_lstm, df_test_lstm]).sort_values([DATE_COL, ID_COL])
X_test_seq, y_test_seq = create_sequences(
    df_test_lstm, FEATURE_COLS, 'mood_label', ID_COL, DATE_COL, SEQUENCE_LENGTH
)

print(f"Train sequences: {X_train_seq.shape} (samples, timesteps, features)")
print(f"Test sequences:  {X_test_seq.shape}")
print(f"Number of classes: {len(class_names)}")

In [ ]:
# --- PyTorch Dataset and DataLoader ---

class MoodSequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.FloatTensor(sequences)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

train_dataset = MoodSequenceDataset(X_train_seq, y_train_seq)
test_dataset = MoodSequenceDataset(X_test_seq, y_test_seq)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)  # keep temporal order
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
# --- LSTM Model Definition ---

class MoodLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(MoodLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # x shape: (batch, seq_len, features)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # Use the last hidden state
        out = self.dropout(h_n[-1])
        out = self.fc(out)
        return out

# Model hyperparameters
INPUT_SIZE = X_train_seq.shape[2]  # number of features
HIDDEN_SIZE = 64
NUM_LAYERS = 2
NUM_CLASSES = len(class_names)
DROPOUT = 0.3
LEARNING_RATE = 0.001
NUM_EPOCHS = 50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = MoodLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT).to(device)
print(f"\nModel architecture:\n{model}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# --- Train the LSTM ---
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

train_losses = []
train_accs = []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for sequences, labels in train_loader:
        sequences, labels = sequences.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item() * sequences.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    scheduler.step(epoch_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

print("\nTraining complete.")

In [ ]:
# --- Training curves ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, color='steelblue')
axes[0].set_title('LSTM Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(train_accs, color='darkorange')
axes[1].set_title('LSTM Training Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')

plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluate LSTM on test set ---
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        outputs = model(sequences)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

y_pred_lstm = np.array(all_preds)
y_test_lstm = np.array(all_labels)

lstm_accuracy = accuracy_score(y_test_lstm, y_pred_lstm)
lstm_f1 = f1_score(y_test_lstm, y_pred_lstm, average='macro')

print("=== LSTM - Test Set Results ===")
print(f"Accuracy: {lstm_accuracy:.4f}")
print(f"F1 (macro): {lstm_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test_lstm, y_pred_lstm, target_names=class_names))

In [ ]:
# --- Confusion matrix for LSTM ---
fig, ax = plt.subplots(figsize=(6, 5))
cm_lstm = confusion_matrix(y_test_lstm, y_pred_lstm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lstm, display_labels=class_names)
disp.plot(ax=ax, cmap='Oranges', values_format='d')
ax.set_title('LSTM - Confusion Matrix')
plt.tight_layout()
plt.show()

## 5. Comparison of Classifiers

In [ ]:
# --- Side-by-side comparison ---
print("=" * 55)
print(f"{'Metric':<25} {'Random Forest':>12} {'LSTM':>12}")
print("=" * 55)
print(f"{'Accuracy':<25} {rf_accuracy:>12.4f} {lstm_accuracy:>12.4f}")
print(f"{'F1 (macro)':<25} {rf_f1:>12.4f} {lstm_f1:>12.4f}")
print("=" * 55)

In [ ]:
# --- Confusion matrices side by side ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=class_names)
disp_rf.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Random Forest')

disp_lstm = ConfusionMatrixDisplay(confusion_matrix=cm_lstm, display_labels=class_names)
disp_lstm.plot(ax=axes[1], cmap='Oranges', values_format='d')
axes[1].set_title('LSTM')

fig.suptitle('Confusion Matrices - Classifier Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Bar chart comparison ---
metrics = ['Accuracy', 'F1 (macro)']
rf_scores = [rf_accuracy, rf_f1]
lstm_scores = [lstm_accuracy, lstm_f1]

x = np.arange(len(metrics))
width = 0.3

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, rf_scores, width, label='Random Forest', color='steelblue')
bars2 = ax.bar(x + width/2, lstm_scores, width, label='LSTM', color='darkorange')

ax.set_ylabel('Score')
ax.set_title('Classifier Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1)
ax.legend()

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

### Interpretation

**TODO: Fill in after running the notebook with actual data.**

- Which classifier performed better and why?
- Did the LSTM benefit from temporal patterns, or was the tabular Random Forest sufficient?
- Are there specific mood classes that are harder to predict? (check confusion matrices)
- How do the feature importances from Random Forest relate to domain knowledge about mood prediction?
- Consider whether the class boundaries (tercile-based) are clinically meaningful, or if different binning strategies might yield better results.
- Discuss potential improvements: more data, different sequence lengths for LSTM, ensemble methods, etc.